# Preprocessing task 

## Nettoyage des tables 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)
NA = ["NA", "NULL", "None", "Not Applicable"]


### Bike types 
il faut :
- garder les 30 types vides avec unknown.
- supprimer les colones max_range (vide a 88%) et battery_capacity (vide a 82%), vehicule_image et description sont inutiles
- Ajouter is_electric 

In [2]:
bike_types = pd.read_csv(RAW / "bike_types.csv", na_values=NA)

bike_types = bike_types.drop(columns=["vehicle_image", "description", "max_range", "battery_capacity"])
bike_types[["form_factor", "propulsion_type"]] = bike_types[["form_factor", "propulsion_type"]].fillna("unknown")
bike_types["is_electric"] = bike_types["propulsion_type"].map(
    {"electric_assist": True, "human": False}).astype("boolean")   # unknown -> <NA>

bike_types.to_parquet(INTERIM / "bike_types.parquet", index=False)


### Bikes :
on a computer_is == 0 pour 13 136, il faut remplacer par NaN et ajouter has_computer

In [3]:
bikes = pd.read_csv(RAW / "bikes.csv", na_values=NA)

bikes["computer_id"] = bikes["computer_id"].replace(0, pd.NA).astype("Int64")
bikes["has_computer"] = bikes["computer_id"].notna()

bikes.to_parquet(INTERIM / "bikes.parquet", index=False)


### Cities : 
- 82 villes sans aucune donnée => il faut les garder dans la table avec un indicateur has_trips

- fuseaux incohérents : 12 villes ES en Europe/Berlin, DE en Europe/Rome, IT, MT, PL et SE en Europe/Berlin => corriger le fuseau à partir du pays (ES → Europe/Madrid, etc.), en gardant Atlantic/Canary pour les Canaries.

In [4]:
cities = pd.read_csv(RAW / "cities.csv", na_values=NA)

country_tz = {"AT": "Europe/Vienna", "CH": "Europe/Zurich", "CY": "Asia/Nicosia",
              "CZ": "Europe/Prague", "DE": "Europe/Berlin", "ES": "Europe/Madrid",
              "GB": "Europe/London", "HR": "Europe/Zagreb", "IT": "Europe/Rome",
              "LV": "Europe/Riga", "MT": "Europe/Malta", "PL": "Europe/Warsaw",
              "SE": "Europe/Stockholm", "SI": "Europe/Ljubljana", "SK": "Europe/Bratislava"}
cities["timezone_raw"] = cities["timezone"]
canary = cities["timezone"] == "Atlantic/Canary"
cities.loc[~canary, "timezone"] = cities.loc[~canary, "country"].map(country_tz)

trip_cities = pd.read_csv(RAW / "trips_small.csv", usecols=["city_id"])["city_id"].unique()
cities["has_trips"] = cities["id"].isin(trip_cities)

cities.to_parquet(INTERIM / "cities.parquet", index=False)


In [5]:
# vérification : fuseaux avant / après correction
pd.crosstab(cities["timezone_raw"], cities["timezone"])

timezone,Asia/Nicosia,Atlantic/Canary,Europe/Berlin,Europe/Bratislava,Europe/Ljubljana,Europe/London,Europe/Madrid,Europe/Malta,Europe/Prague,Europe/Riga,Europe/Rome,Europe/Stockholm,Europe/Vienna,Europe/Warsaw,Europe/Zagreb,Europe/Zurich
timezone_raw,,,,,,,,,,,,,,,,
Asia/Istanbul,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Atlantic/Canary,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Europe/Berlin,0,0,90,0,10,0,12,1,25,0,1,1,24,1,19,0
Europe/Bratislava,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
Europe/Brussels,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,13
Europe/London,0,0,0,0,0,9,0,0,0,0,0,0,0,0,0,0
Europe/Riga,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
Europe/Rome,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Europe/Warsaw,0,0,0,0,0,0,0,0,0,0,0,0,0,52,0,0


In [6]:
cities.loc[cities["country"] == "CY", ["name", "timezone_raw", "timezone", "lat", "lon"]]

,name,timezone_raw,timezone,lat,lon
70,Nicosia,Asia/Istanbul,Asia/Nicosia,35.1991,33.4535
213,Limassol,Asia/Istanbul,Asia/Nicosia,34.6823,33.0464


### Station :
- Supprimer la colonne rack_locks : elle contient uniquement des valeurs manquantes (100 %).

- Traiter les valeurs négatives de special_racks : remplacer les valeurs inférieures à 0 (32 stations) par NaN, car elles sont considérées comme invalides.

- Identifier les stations virtuelles : ajouter une colonne booléenne is_virtual pour identifier les stations ayant bike_racks == 0 (1 219 stations).

- Gérer les valeurs manquantes de terminal_type : remplacer les valeurs manquantes (587 stations) par unknown.

- Nettoyer les coordonnées géographiques :
- Supprimer les coordonnées géographiques impossibles ou invalides.
- Conserver les 42 stations situées à plus de 50 km du centre de leur ville, car elles peuvent correspondre à des systèmes régionaux légitimes (ex. : Wolgast, Prora).
- Vérifier les modifications : contrôler les valeurs manquantes, les valeurs aberrantes et la cohérence des données après nettoyage.

In [7]:
stations = pd.read_csv(RAW / "stations.csv", na_values=NA)

stations = stations.drop(columns="rack_locks")
stations.loc[stations["special_racks"] < 0, "special_racks"] = np.nan
stations["terminal_type"] = stations["terminal_type"].fillna("unknown")
stations["is_virtual"] = stations["bike_racks"] == 0
stations["capacity"] = stations["bike_racks"] + stations["special_racks"].fillna(0)

# coordonnées impossibles (hors Europe ou Canaries)
bad_coords = ~stations["lat"].between(27, 72) | ~stations["lon"].between(-25, 45)
print("stations aux coordonnées impossibles :", bad_coords.sum())
stations = stations[~bad_coords]

stations.to_parquet(INTERIM / "stations.parquet", index=False)


stations aux coordonnées impossibles : 1


In [8]:
# vérification
print(stations.shape)
print("special_racks < 0 :", (stations["special_racks"] < 0).sum())
print("stations virtuelles :", stations["is_virtual"].sum())
stations[["lat", "lon"]].describe()

(2931, 12)
special_racks < 0 : 0
stations virtuelles : 1219


,lat,lon
count,2931.000000,2931.000000
mean,49.286940,13.174015
std,4.274932,6.914737
min,27.740583,-15.758277
25%,47.961091,8.158378
50%,50.290498,14.314438
75%,51.602480,17.490702
max,57.005195,33.413632


In [9]:
bikes      = pd.read_parquet(INTERIM / "bikes.parquet")
bike_types = pd.read_parquet(INTERIM / "bike_types.parquet")
cities     = pd.read_parquet(INTERIM / "cities.parquet")      # fuseaux déjà corrigés
stations   = pd.read_parquet(INTERIM / "stations.parquet")


## fonctions pour l'heure locale

In [10]:
def to_local(utc, tz):
    """Convertit une série UTC en heure locale, ligne par ligne selon le fuseau."""
    pieces = [utc[tz == z].dt.tz_convert(z).dt.tz_localize(None)
              for z in tz.dropna().unique()]
    return pd.concat(pieces).reindex(utc.index)

def add_time_features(df, col):
    df["hour"]       = df[col].dt.hour
    df["weekday"]    = df[col].dt.day_of_week        # 0 = lundi
    df["is_weekend"] = df["weekday"] >= 5
    df["month"]      = df[col].dt.to_period("M").astype(str)
    return df


trips

In [11]:
trips = pd.read_csv(RAW / "trips_small.csv", na_values=NA)
n_before = len(trips)

# temps
trips["time_start"] = pd.to_datetime(trips["time_start"], unit="s", utc=True)

# stations : identifiants entiers, free-floating, stations inconnues
for col in ["station_id_start", "station_id_end"]:
    trips[col] = trips[col].astype("Int64")
trips["free_floating_start"] = trips["station_id_start"].isna()
trips["free_floating_end"]   = trips["station_id_end"].isna()
trips["start_station_known"] = trips["station_id_start"].isin(stations["id"])
trips["end_station_known"]   = trips["station_id_end"].isin(stations["id"])

# fusion avec le type de vélo
bike_info = (bikes[["id", "bike_type_id", "has_computer"]]
             .merge(bike_types[["id", "propulsion_type", "form_factor", "is_electric"]]
                    .rename(columns={"id": "bike_type_id"}),
                    on="bike_type_id", how="left")
             .rename(columns={"id": "bike_id"}))
trips = trips.merge(bike_info, on="bike_id", how="left")

# fusion avec la ville (fuseau, pays, règle de retour en station)
trips = trips.merge(cities[["id", "timezone", "country", "return_to_official_only"]]
                    .rename(columns={"id": "city_id"}),
                    on="city_id", how="left")

assert len(trips) == n_before, "la fusion a dupliqué des lignes"

# batterie
trips["battery_inconsistent"] = ((trips["is_electric"] == False)
                                 & trips["battery_start"].notna()).fillna(False)
trips["battery_used"] = trips["battery_start"] - trips["battery_end"]

# vitesse
trips["speed_kmh"] = trips["distance"] / trips["duration"] * 3.6

# heure locale
trips["time_start_local"] = to_local(trips["time_start"], trips["timezone"])
trips = add_time_features(trips, "time_start_local")

trips.to_parquet(INTERIM / "trips.parquet", index=False)


In [12]:
print(trips.shape)
print("heure locale manquante :", trips["time_start_local"].isna().sum())
print("trajets 'human' avec batterie :", trips["battery_inconsistent"].sum())
print("stations de départ inconnues :",
      (~trips["free_floating_start"] & ~trips["start_station_known"]).sum())
print("stations d'arrivée inconnues :",
      (~trips["free_floating_end"] & ~trips["end_station_known"]).sum())
trips[["duration", "distance", "speed_kmh", "battery_used"]].describe()


(1161376, 33)
heure locale manquante : 0
trajets 'human' avec batterie : 302
stations de départ inconnues : 16704
stations d'arrivée inconnues : 76040


,duration,distance,speed_kmh,battery_used
count,1.161376e+06,1.161376e+06,1.161376e+06,168518.000000
mean,1.256241e+03,1.791774e+03,7.318860e+00,4.290901
std,1.980838e+03,1.495191e+03,3.303534e+00,6.668139
min,1.210000e+02,5.001365e+01,2.500312e-01,-100.000000
25%,4.800000e+02,8.851685e+02,5.160254e+00,0.000000
50%,7.200000e+02,1.346711e+03,7.535472e+00,3.000000
75%,1.200000e+03,2.238540e+03,9.613301e+00,6.000000
max,2.874000e+04,1.999946e+04,1.504991e+01,100.000000


stations_status

In [13]:
status = pd.read_csv(RAW / "stations_status.csv", na_values=NA)
n_before = len(status)

status["time"] = pd.to_datetime(status["time"], unit="s", utc=True)

# lignes invalides : réservations négatives ou total incohérent
invalid = (status["booked_bikes"] < 0) | \
          (status["bikes"] != status["booked_bikes"] + status["bikes_available_to_rent"])
print("lignes invalides supprimées :", invalid.sum())
status = status[~invalid]

# fusion avec la station (ville, capacité) puis avec la ville (fuseau)
status = status.merge(stations[["id", "city_id", "capacity", "is_virtual"]]
                      .rename(columns={"id": "station_id"}),
                      on="station_id", how="left")
status = status.merge(cities[["id", "timezone"]].rename(columns={"id": "city_id"}),
                      on="city_id", how="left")

# stations avec trop peu de relevés : on les marque sans les supprimer
status["enough_snapshots"] = status.groupby("station_id")["time"].transform("size") >= 10

# heure locale
status["time_local"] = to_local(status["time"], status["timezone"])
status = add_time_features(status, "time_local")

status = status.sort_values(["station_id", "time"]).reset_index(drop=True)
status.to_parquet(INTERIM / "stations_status.parquet", index=False)


lignes invalides supprimées : 2496


vérification de stations_status

In [14]:
print(n_before, "->", len(status))
print("relevés sans station connue :", status["city_id"].isna().sum())
print("en maintenance :", status["maintenance"].mean())
print("stations avec moins de 10 relevés :",
      status.loc[~status["enough_snapshots"], "station_id"].nunique())


2925967 -> 2923471
relevés sans station connue : 8
en maintenance : 0.044644875902651336
stations avec moins de 10 relevés : 84


## Fusion

In [15]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

### Stations + villes

In [16]:
stations_full = stations.merge(
    cities[["id", "name", "country", "timezone", "return_to_official_only", "lat", "lon"]]
    .rename(columns={"id": "city_id", "name": "city_name", "lat": "city_lat", "lon": "city_lon"}),
    on="city_id", how="left")
stations_full["dist_center_km"] = haversine_km(stations_full["lat"], stations_full["lon"],
                                               stations_full["city_lat"], stations_full["city_lon"])
assert len(stations_full) == len(stations)

stations_full.to_parquet(PROCESSED / "stations.parquet", index=False)
stations_full.shape

(2931, 19)

### Trajets + stations de départ et d'arrivée

In [17]:
st_cols = ["id", "is_virtual", "capacity", "dist_center_km"]
trips_full = (trips
    .merge(stations_full[st_cols].add_suffix("_start"),
           left_on="station_id_start", right_on="id_start", how="left")
    .merge(stations_full[st_cols].add_suffix("_end"),
           left_on="station_id_end", right_on="id_end", how="left")
    .drop(columns=["id_start", "id_end"]))
trips_full["same_station"] = trips_full["station_id_start"] == trips_full["station_id_end"]
assert len(trips_full) == len(trips)

trips_full.to_parquet(PROCESSED / "trips.parquet", index=False)
trips_full.shape

(1161376, 40)

### Relevés des stations

In [18]:
n = len(status)
status_full = status.dropna(subset=["city_id"]).copy()   # relevés de stations supprimées au nettoyage
print("relevés supprimés :", n - len(status_full))
status_full["city_id"] = status_full["city_id"].astype("int64")

status_full.to_parquet(PROCESSED / "stations_status.parquet", index=False)
status_full.shape

relevés supprimés : 8


(2923463, 18)